In [1]:
from src import *
import numpy as np
from tqdm import tqdm
from time import sleep


In [2]:
sampling_rate = 20 #Hz
freq_list = np.array([0.1 , 0.12, 0.14, 0.16, 0.18, 0.2 , 0.22, 0.24, 0.26, 0.28, 0.3, 0.32, 0.34, 0.36, 0.38, 0.4])

pic_time = (1 / (sampling_rate * freq_list) / 2 * 1e6).astype(int)

In [3]:
with qCMOS() as qcmos:
    with DMD() as dmd:
        for i, picture_time in enumerate(pic_time):
            A = 5
            samples = 50
            repeat = 200

            interval = 50e-3
            exposure_time = 500e-6 #500 us

            dmd.ez_load_seq([dmd.ez_single_pixel(0), dmd.ez_single_pixel(A)], picture_time)

            qcmos.ez_exposure_time(exposure_time)
            qcmos.ez_triggersource_masterpluse(samples, interval)
            qcmos.ez_roi(**SPADE.ROI)

            data, timestamp = [], []
            for _ in tqdm(range(repeat)):
                qcmos.buf_alloc(samples)
                qcmos.cap_snapshot()

                dmd.Run()
                sleep(1e-6)
                qcmos.cap_firetrigger()

                qcmos.ez_wait_capture()

                qcmos.cap_stop()
                dmd.Halt()

                data_, timestamp_ = [], []
                for frame in range(samples):
                    framedata_ = qcmos.ez_read_buf(frame)
                    data_.append(framedata_[0])
                    timestamp_.append(framedata_[1])

                qcmos.buf_release()

                data.append(data_)
                timestamp.append(timestamp_)

            np.save(f'./__temp__/spade_{freq_list[i]}.npy', np.array(data))
            np.save(f'./__temp__/timestamp_{freq_list[i]}.npy', np.array(timestamp))

Loading library: c:\users\zzbn\Desktop\freqest\src\api/x64/alp4395.dll
DMD found, resolution = 1024 x 768.


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


exited
exited
